# 04 — Cross-dataset explanation consistency (H2, H3 + statistical validation)

Loads the per-seed SHAP rankings from notebook 03 and computes:
- Spearman rho, Kendall tau, Jaccard@k, RBO, MAD, and the aggregate ECS per class
- the same-dataset upper bound (two reseeded source models) and random-permutation
  lower bound (Sec 4.3)
- a Wilcoxon signed-rank test of observed ECS vs the random baseline


In [ ]:
import sys, os, pickle
sys.path.append(os.path.abspath('..'))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.explain.consistency import compute_all_class_consistency, explanation_consistency_score
from src.eval.baselines import random_permutation_baseline, wilcoxon_vs_random, summarise_across_seeds
from src.data.schema import SHARED_FEATURES

with open('../configs/default.yaml') as f:
    cfg = yaml.safe_load(f)
with open('../data/processed/class_names.yaml') as f:
    class_names = yaml.safe_load(f)

seeds = cfg['seeds'][:cfg['n_seeds']]
per_seed = {}
for seed in seeds:
    with open(f'../results/tables/shap_rankings_seed{seed}.pkl', 'rb') as f:
        per_seed[seed] = pickle.load(f)

## Cross-dataset consistency per class, per seed

In [ ]:
rows = []
for seed in seeds:
    consistency = compute_all_class_consistency(
        per_seed[seed]['source'], per_seed[seed]['target'], SHARED_FEATURES, cfg['shap']['top_k']
    )
    for cls, metrics in consistency.items():
        rows.append({'seed': seed, 'class': cls, **metrics})
consistency_df = pd.DataFrame(rows)
consistency_df.to_csv('../results/tables/cross_dataset_consistency_per_seed.csv', index=False)
consistency_df

## Mean ± std across seeds, per class (headline table)

In [ ]:
summary = consistency_df.groupby('class').agg(['mean', 'std'])
summary.to_csv('../results/tables/cross_dataset_consistency_summary.csv')
summary

## Random-permutation lower bound + Wilcoxon significance test

In [ ]:
random_ecs, observed_ecs = [], []
for seed in seeds:
    for cls in class_names:
        if cls not in per_seed[seed]['source']:
            continue
        obs = explanation_consistency_score(
            per_seed[seed]['source'][cls]['ranking'], per_seed[seed]['target'][cls]['ranking'],
            per_seed[seed]['source'][cls]['importance'], per_seed[seed]['target'][cls]['importance'],
            SHARED_FEATURES, cfg['shap']['top_k'],
        )
        rand = random_permutation_baseline(
            per_seed[seed]['source'][cls]['ranking'], per_seed[seed]['source'][cls]['importance'],
            SHARED_FEATURES, seed, cfg['shap']['top_k'],
        )
        observed_ecs.append(obs['explanation_consistency_score'])
        random_ecs.append(rand['explanation_consistency_score'])

# Pairs are seeds x classes (5 x 4 = 20). Do NOT collapse to classes alone:
# a one-sided signed-rank test cannot reach p < 0.05 with fewer than 6 pairs.
test_result = wilcoxon_vs_random(observed_ecs, random_ecs)
print('Observed ECS:', summarise_across_seeds(observed_ecs))
print('Random baseline ECS:', summarise_across_seeds(random_ecs))
print('Wilcoxon signed-rank (observed > random):', test_result)
if test_result.get('underpowered'):
    print('WARNING: fewer than 6 pairs — this test cannot reach significance.')

## Same-dataset upper bound (Sec 4.3)

Consistency between two *independently seeded* models trained on the **same**
dataset. This is the ceiling: cross-dataset consistency can never sensibly exceed
the agreement two runs on identical data achieve with each other. Without this
anchor a raw rho of, say, 0.55 is uninterpretable — it could be near-perfect
transfer or near-total failure depending on where the ceiling sits.

In [ ]:
from src.eval.baselines import same_dataset_upper_bound
from itertools import combinations

upper_rows = []
for seed_a, seed_b in combinations(seeds, 2):
    ub = same_dataset_upper_bound(
        per_seed[seed_a]['source'], per_seed[seed_b]['source'], SHARED_FEATURES, cfg['shap']['top_k'])
    for cls, m in ub.items():
        upper_rows.append({'seed_pair': f'{seed_a}v{seed_b}', 'class': cls, **m})
upper_df = pd.DataFrame(upper_rows)
upper_df.to_csv('../results/tables/same_dataset_upper_bound.csv', index=False)

print('Upper bound (same dataset, reseeded)  ECS:',
      round(upper_df['explanation_consistency_score'].mean(), 4),
      '+/-', round(upper_df['explanation_consistency_score'].std(), 4))
print('Observed  (cross-dataset)             ECS:',
      round(consistency_df['explanation_consistency_score'].mean(), 4),
      '+/-', round(consistency_df['explanation_consistency_score'].std(), 4))
print('Lower bound (random permutation)      ECS:',
      round(np.mean(random_ecs), 4))

### Headline framing

Report cross-dataset consistency **as a fraction of the achievable ceiling**, not as a
bare correlation. `(observed - random) / (upper_bound - random)` is the quantity that
actually answers the research question.

In [ ]:
obs_mean = consistency_df['explanation_consistency_score'].mean()
ub_mean = upper_df['explanation_consistency_score'].mean()
lb_mean = float(np.mean(random_ecs))

normalised = (obs_mean - lb_mean) / (ub_mean - lb_mean) if ub_mean > lb_mean else float('nan')
print(f'Normalised cross-dataset consistency: {normalised:.3f}')
print('  1.0 = explanations transfer as well as a same-dataset reseed')
print('  0.0 = explanations transfer no better than random')

## Plot: consistency by attack class (H3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
means = consistency_df.groupby('class')['explanation_consistency_score'].mean()
stds = consistency_df.groupby('class')['explanation_consistency_score'].std()
ax.bar(means.index, means.values, yerr=stds.values, capsize=4)
ax.set_ylabel('Explanation Consistency Score')
ax.set_title('Cross-dataset SHAP consistency by attack class')
plt.xticks(rotation=20)
plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/consistency_by_class.png', dpi=150)
plt.show()